# Hour 1 · Corpora and data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/1a_corpora_and_data.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F1a_corpora_and_data.ipynb)


**Goal:** see how a clay tablet becomes a *programmatically accessible corpus* — not an e-book, but a graph of objects (tablet → line → word → sign) with features.

We use the **Copenhagen Ugaritic Corpus (CUC)**, served as JSONL from HuggingFace and loaded with one function. Licence: **CC BY-NC 4.0** (educational use).

*Reading:* `docs/02-corpora-and-data.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. One tablet, two scripts
Every line carries both a Latin transliteration (`lines`) and the actual **cuneiform** (`ugaritic`), plus a reference (`refs`).

In [ ]:
t = [x for x in texts if x["ktu"] == "1.3"][0]   # Baal Cycle
for ref, lat, cun in zip(t["refs"][:6], t["lines"][:6], t["ugaritic"][:6]):
    print(f'{ref:14s} {lat:28s} {cun}')

## 2. The corpus is a graph of object types
In Text-Fabric the CUC objects are **tablet · column · line · word · sign**. Our loader keeps tablets, lines, word tokens, and signs.

In [ ]:
print("tablets :", len(texts))
print("lines   :", sum(len(t["lines"]) for t in texts))
print("words   :", sum(len(t["tokens"]) for t in texts))
from data.loader import sign_counts
print("signs   :", sum(sign_counts(texts).values()))

## 3. Browse by genre
Genre labels are heuristic (KTU number + known tablets).

In [ ]:
from data.loader import texts_by_genre
for g, items in sorted(texts_by_genre(texts).items(), key=lambda kv: -len(kv[1])):
    print(f'{g:20s} {len(items):3d}  e.g. {", ".join(x["ktu"] for x in items[:5])}')

## 4. Simple queries
A corpus lets us *ask questions* in code.

In [ ]:
# find every tablet that contains a given word form
TARGET = "mlk"   # "king"
hits = [t["ktu"] for t in texts if TARGET in t["tokens"]]
print(f'{TARGET!r} appears in {len(hits)} tablets:', hits[:15], "...")

In [ ]:
# print every line of one tablet that contains the word
ktu = "1.4"
one = [t for t in texts if t["ktu"] == ktu][0]
for ref, line in zip(one["refs"], one["lines"]):
    if TARGET in line.replace(".", " ").split():
        print(f'{ref}: {line}')

## 5. Discussion
A corpus is a **graph of objects and features**, queryable in code. CUC shares the Text-Fabric model with **BHSA** (Hebrew Bible) and **DSS** — the same queries port across corpora.

> **With full Text-Fabric** (`pip install text-fabric`; `use('DT-UCPH/cuc')`) you also get sign-level features: emendation, certainty, alternative readings.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [ ]:
# Change the word and re-run. Try: "ilm" (gods), "ank" (I), "bʿl" (Baal), "ytn" (he gave)
MY_WORD = "ilm"
tablets = [t["ktu"] for t in texts if MY_WORD in t["tokens"]]
print(f'{MY_WORD!r} occurs in {len(tablets)} tablets:', tablets[:20])
# Hint: words are case-sensitive transliteration — ʿ ʾ š ḥ ṯ ġ are distinct letters.